## **09_layer_norm: Keeping it Stable: Layer Normalization**

In the last chapter, we introduced the "express lane" of our Transformer Block: the residual connection.  
But a highway with no rules can lead to chaos.  
We need a **stabilizer** to ensure the data flowing through our network remains well-behaved.

### The Problem: Internal Covariate Shift

As data flows through a deep network, the distribution of activations at each layer is **constantly changing** during training. The mean, the standard deviation — it's all over the place.

Each layer is trying to learn, but the target it's aiming for is **constantly moving**.  
It's like trying to shoot arrows at a target strapped to a bucking bronco.

### The Solution: Layer Normalization

For **each individual token's vector**, it performs these steps independently:

1. Calculate the **mean** ($\mu$) and **variance** ($\sigma^2$) across the `C` dimension
2. **Normalize**: $\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$ → forces mean=0, std=1
3. Apply **learnable** parameters: $y = \gamma \cdot \hat{x} + \beta$ (gives the model back control)

### Step 1: The Input Vector

Imagine this is a token's vector after a residual connection.  
Its values have shifted away from a clean distribution.

In [ ]:
import torch
import torch.nn as nn

# A sample vector for one token, shape (B, T, C)
x_token = torch.tensor([[[0.3, -0.2, 0.8, 0.5]]])
print("Input to LayerNorm (x):\n", x_token)

mean = x_token.mean(dim=-1, keepdim=True)
std = x_token.std(dim=-1, keepdim=True)
print(f"\nMean of input: {mean.item():.2f}")
print(f"Std Dev of input: {std.item():.2f}")

The vector is **not** centered at zero and its values are not scaled to a standard deviation of one.  
It's a moving target.

### Step 2: Normalization (The Core $\hat{x}$ Calculation)

Force the vector to have **mean=0** and **std=1**.

In [ ]:
epsilon = 1e-5

# Manually normalize
x_hat = (x_token - mean) / torch.sqrt(std**2 + epsilon)

print("Normalized vector (x_hat):\n", x_hat.data.round(decimals=2))
print(f"\nMean of x_hat: {x_hat.mean().item():.2f}")
print(f"Std Dev of x_hat: {x_hat.std().item():.2f}")

Mean = 0, Std = 1. The target is no longer moving.

### Step 3: Applying the Learnable Parameters ($\gamma$ and $\beta$)

After forcing a standard normal distribution, the model can **learn** the optimal scale and shift for the next layer.  

`nn.LayerNorm` creates these automatically:  
+ `weight` ($\gamma$): initialized to **all ones**
+ `bias` ($\beta$): initialized to **all zeros**

Initially: multiplying by 1 and adding 0 changes nothing.  
But during training, the model learns optimal values.

In [ ]:
C = 4
ln = nn.LayerNorm(C)
print("--- Initial Parameters ---")
print(f"gamma (weight): {ln.weight.data}")
print(f"beta (bias):    {ln.bias.data}")

In [ ]:
# Pretend the model has learned optimal values
gamma = torch.tensor([1.5, 1.0, 1.0, 1.0])
beta = torch.tensor([0.5, 0.0, 0.0, 0.0])

# Apply gamma and beta
y = gamma * x_hat + beta

print("--- After Applying Learned Gamma and Beta ---")
print("Final output (y):\n", y.data.round(decimals=2))
print(f"\nMean of y: {y.mean().item():.2f}")
print(f"Std Dev of y: {y.std().item():.2f}")

The model has used $\gamma$ and $\beta$ to find the most useful distribution for the next layer.

### Verify: `nn.LayerNorm` does it all in one call

In [ ]:
ln = nn.LayerNorm(C)
output = ln(x_token)
print("nn.LayerNorm output:\n", output.data.round(decimals=2))
print(f"Mean: {output.mean().item():.4f}")
print(f"Std:  {output.std().item():.2f}")

### Pre-Norm vs. Post-Norm

The placement of LayerNorm relative to the residual connection is an important choice.

| Feature | Pre-Norm (GPT-2 style) | Post-Norm (Original Transformer) |
| :--- | :--- | :--- |
| **Equation** | `x + Sublayer(LayerNorm(x))` | `LayerNorm(x + Sublayer(x))` |
| **Stability** | More stable training for deep networks | Can be harder to train; needs LR warm-up |
| **Code** | `x = x + self.attn(self.ln_1(x))` | `x = self.ln_1(x + self.attn(x))` |

**Why GPT-2 uses Pre-Norm:**  
By normalizing the input *before* the sub-layer, we keep values in check and prevent explosions.  
It just makes training smoother and more robust.

With residual connections providing the "express lane" and layer normalization acting as the "stabilizer", we now have all the necessary components to assemble a complete Transformer Block.